# Chuẩn bị dữ liệu: lọc video trùng → chia train / val / test

**Vai trò các bộ dữ liệu**

| Bộ | Vai trò |
|---|---|
| **RWF-2000** (bắt buộc) | Train + val (tách từ thư mục `train`), **test chính** (thư mục `val` của tác giả) |
| **RLVS** (tùy chọn) | Test chéo 1 |
| **Hockey Fights** (tùy chọn) | Test chéo 2 |

**Các bước (chạy tuần tự từng ô, xem kết quả rồi mới chạy tiếp)**

1. Quét video, đọc nhãn, kiểm tra video lỗi, tạo dấu vân tay mỗi video
2. Chọn ngưỡng phát hiện trùng bằng cách xem trực quan các cặp video
3. Gom nhóm video trùng trong RWF-2000
4. Tách tập test chính thức, loại clip test trùng với train
5. Chia train / val theo nhóm (không để clip trùng nằm ở hai tập)
6. Loại clip của RLVS / Hockey trùng với train/val của RWF-2000
7. Kiểm tra rò rỉ dữ liệu
8. Lưu file CSV
9. Kiểm tra thử đọc frame

> RWF-2000 cấm phân phối lại. Chỉ chạy trên máy cá nhân hoặc Drive riêng tư, không đưa lên Kaggle công khai.

## Bước 0. Thư viện và cấu hình

In [ ]:
# Nếu thiếu thư viện, bỏ dấu # ở dòng dưới rồi chạy một lần:
# %pip install opencv-python numpy pandas scikit-learn tqdm matplotlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
# ====== CHỈNH Ở ĐÂY ======
RWF_DIR    = Path(r"D:\TaiLieu\2026-2027\ATTT\dataset\RWF-2000")
RLVS_DIR   = Path(r"D:\TaiLieu\2026-2027\ATTT\dataset\Real Life Violence Dataset")
HOCKEY_DIR = Path(r"D:\TaiLieu\2026-2027\ATTT\dataset\Hockey")
# Bộ nào chưa có thì đặt = None

OUT_DIR   = Path("manifests")   # nơi lưu file CSV kết quả
CACHE_DIR = Path("cache")       # lưu kết quả quét để chạy lại cho nhanh
USE_CACHE = True                # đổi dữ liệu thì xóa thư mục cache hoặc đặt False

SEED = 42
VAL_RATIO = 0.15                # tỉ lệ validation tách từ phần train của RWF-2000
OFFICIAL_TEST_FOLDER = "val"    # thư mục của tác giả RWF-2000 dùng làm tập test
# =========================

OUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

DATASET_DIRS = {"rwf": RWF_DIR, "rlvs": RLVS_DIR, "hockey": HOCKEY_DIR}
DATASET_DIRS = {k: v for k, v in DATASET_DIRS.items() if v is not None and Path(v).exists()}
print("Bộ dữ liệu tìm thấy:", list(DATASET_DIRS))
assert "rwf" in DATASET_DIRS, "Không tìm thấy thư mục RWF-2000, kiểm tra lại RWF_DIR"

## Bước 1. Quét video, đọc nhãn, tạo dấu vân tay

Mỗi video được:
- đọc nhãn từ tên thư mục (`Fight`/`Violence` = 1, `NonFight`/`NonViolence` = 0; Hockey dùng tiền tố tên file `fi*`/`no*`)
- kiểm tra mở được, đủ số frame
- lấy 5 frame đều nhau, thu nhỏ rồi mã hóa thành 320 bit (**dấu vân tay**). Hai video có dấu vân tay gần nhau là video trùng hoặc gần trùng.

Bước này chạy lâu nhất (vài phút), kết quả được lưu cache.

In [ ]:
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
MIN_FRAMES, N_HASH_FRAMES, HASH_SIZE = 16, 5, 8
NEG_KEYS = ("nonviolence", "non_violence", "non-violence", "nonfight", "non_fight", "non-fight")
POS_KEYS = ("violence", "fight")

def get_label(path, root, dataset):
    parts = [p.lower() for p in path.relative_to(root).parts[:-1]]
    for p in parts:
        if any(k in p for k in NEG_KEYS):
            return 0
    for p in parts:
        if any(k in p for k in POS_KEYS):
            return 1
    if dataset == "hockey":
        name = path.name.lower()
        if name.startswith("fi"): return 1
        if name.startswith("no"): return 0
    return None

def dhash(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (HASH_SIZE + 1, HASH_SIZE))
    return (gray[:, 1:] > gray[:, :-1]).flatten().astype(np.uint8)

def inspect_video(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None, "khong_mo_duoc"
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if total < MIN_FRAMES:
        cap.release(); return None, f"qua_ngan_{total}_frame"
    hashes = []
    for idx in np.linspace(0, total - 1, N_HASH_FRAMES).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            cap.release(); return None, "doc_frame_loi"
        hashes.append(dhash(frame))
    cap.release()
    info = dict(frames=total, fps=round(fps, 2), width=w, height=h,
                duration_s=round(total / fps, 2) if fps > 0 else None)
    return info, np.concatenate(hashes)

def scan_dataset(name, root):
    cache = CACHE_DIR / f"{name}.pkl"
    if USE_CACHE and cache.exists():
        print(f"[{name}] dùng cache")
        return pd.read_pickle(cache)
    root = Path(root)
    files = sorted(p for p in root.rglob("*") if p.suffix.lower() in VIDEO_EXTS)
    rows, fps, bad = [], [], []
    for p in tqdm(files, desc=f"Quét {name}"):
        rel = p.relative_to(root)
        label = get_label(p, root, name)
        if label is None:
            bad.append(dict(dataset=name, path=str(rel), reason="khong_xac_dinh_nhan")); continue
        info, res = inspect_video(p)
        if info is None:
            bad.append(dict(dataset=name, path=str(rel), reason=res)); continue
        orig = next((s.lower() for s in rel.parts[:-1] if s.lower() in ("train", "val", "test")), "")
        rows.append(dict(dataset=name, path=str(rel), abs_path=str(p), label=label, orig_split=orig, **info))
        fps.append(res)
    n_bits = N_HASH_FRAMES * HASH_SIZE * HASH_SIZE
    fp = np.stack(fps) if fps else np.zeros((0, n_bits), dtype=np.uint8)
    res = (pd.DataFrame(rows), fp, pd.DataFrame(bad))
    pd.to_pickle(res, cache)
    return res

In [ ]:
data = {name: scan_dataset(name, root) for name, root in DATASET_DIRS.items()}
for name, (df, fp, bad) in data.items():
    print(f"{name:7s} hợp lệ: {len(df):5d} | bị loại: {len(bad):3d}")

In [ ]:
# Thống kê số clip theo nhãn (1 = đánh nhau) và theo thư mục gốc
for name, (df, fp, bad) in data.items():
    print(f"=== {name} ===")
    print(pd.crosstab(df["orig_split"].replace("", "(không có)"), df["label"]).to_string())
    print()

In [ ]:
# Video bị loại (nếu có)
bad_all = pd.concat([b for _, _, b in data.values()], ignore_index=True)
print(f"Tổng video bị loại: {len(bad_all)}")
display(bad_all.head(20)) if len(bad_all) else print("Không có video lỗi")

In [ ]:
# Độ dài và độ phân giải: kiểm tra các bộ có khác nhau nhiều không
for name, (df, fp, bad) in data.items():
    print(f"=== {name} ===")
    print(df[["frames", "fps", "width", "height", "duration_s"]].describe().loc[["min", "50%", "max"]].round(1).to_string())
    print()

## Bước 2. Chọn ngưỡng phát hiện trùng

**Khoảng cách** giữa hai video = số bit khác nhau trong dấu vân tay (trên 320 bit). Càng nhỏ càng giống.

- Ô đầu: biểu đồ khoảng cách đến video **giống nhất** của mỗi video trong RWF-2000.
- Ô sau: hiển thị ảnh các cặp theo từng khoảng cách. **Bạn nhìn bằng mắt** xem đến khoảng cách nào thì hai ảnh còn là cùng một cảnh quay, rồi đặt ngưỡng ở ô cuối.

In [ ]:
rwf, fp_rwf, _ = data["rwf"]

def hamming(a, b):
    a = a.astype(np.float32); b = b.astype(np.float32)
    return a @ (1 - b).T + (1 - a) @ b.T

D = hamming(fp_rwf, fp_rwf)          # ma trận khoảng cách RWF x RWF
np.fill_diagonal(D, 1e9)             # bỏ so video với chính nó
nn = D.min(axis=1)

plt.figure(figsize=(8, 3.5))
plt.hist(nn, bins=40)
plt.xlabel("Khoảng cách đến video giống nhất (số bit khác, tối đa 320)")
plt.ylabel("Số video")
plt.title("RWF-2000: phân bố khoảng cách gần nhất")
plt.show()
print("Số video có bạn gần giống với khoảng cách <=", end=" ")
for t in (10, 20, 30, 40, 50):
    print(f"{t}: {(nn <= t).sum()}", end=" | ")

In [ ]:
def read_mid_frame(path):
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2)
    ok, fr = cap.read(); cap.release()
    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if ok else np.zeros((60, 80, 3), np.uint8)

def preview_pairs(df, D, bins=((0, 10), (10, 20), (20, 30), (30, 40), (40, 50), (50, 60)), per_bin=2, seed=0):
    rng = np.random.default_rng(seed)
    shown = []
    for lo, hi in bins:
        ii, jj = np.where(np.triu((D >= lo) & (D < hi), k=1))
        if len(ii) == 0:
            print(f"Khoảng cách {lo}-{hi}: không có cặp nào"); continue
        pick = rng.choice(len(ii), size=min(per_bin, len(ii)), replace=False)
        for k in pick:
            shown.append((ii[k], jj[k], D[ii[k], jj[k]], len(ii)))
    if not shown:
        print("Không có cặp nào để xem"); return
    fig, axes = plt.subplots(len(shown), 2, figsize=(7, 2.6 * len(shown)))
    axes = np.atleast_2d(axes)
    for r, (i, j, d, tot) in enumerate(shown):
        for c, idx in enumerate((i, j)):
            axes[r, c].imshow(read_mid_frame(df.loc[idx, "abs_path"]))
            axes[r, c].axis("off")
            axes[r, c].set_title(Path(df.loc[idx, "path"]).name[:28], fontsize=8)
        axes[r, 0].text(0, -0.15, f"khoảng cách = {int(d)}", transform=axes[r, 0].transAxes, fontsize=9, color="red")
    plt.tight_layout(); plt.show()

preview_pairs(rwf, D)

In [ ]:
# Sau khi xem ảnh: đặt ngưỡng. Mặc định 40.
# - Nếu cặp ở khoảng cách 30-40 đã là hai cảnh KHÁC nhau  -> giảm xuống (ví dụ 25)
# - Nếu cặp ở khoảng cách 40-50 vẫn là CÙNG cảnh quay      -> tăng lên (ví dụ 50)
THR = 40
print("Ngưỡng đã chọn:", THR)

## Bước 3. Gom nhóm video trùng trong RWF-2000

Các video có khoảng cách ≤ ngưỡng được nối vào cùng một **nhóm**. Khi chia tập, cả nhóm luôn đi cùng nhau.

In [ ]:
def group_by_threshold(D, thr):
    n = len(D)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    ii, jj = np.where(np.triu(D <= thr, k=1))
    for i, j in zip(ii, jj):
        ri, rj = find(i), find(j)
        if ri != rj: parent[rj] = ri
    roots = np.array([find(i) for i in range(n)])
    _, groups = np.unique(roots, return_inverse=True)
    return groups, len(ii)

groups, n_pairs = group_by_threshold(D, THR)
rwf = rwf.copy()
rwf["group"] = groups
sizes = rwf.groupby("group").size()
print(f"Số cặp trùng: {n_pairs}")
print(f"Số nhóm: {rwf['group'].nunique()} / {len(rwf)} video")
print(f"Nhóm có từ 2 video trở lên: {(sizes > 1).sum()} (chứa {sizes[sizes > 1].sum()} video)")
print(f"Kích thước nhóm lớn nhất: {sizes.max()}")
print("\nVí dụ vài nhóm trùng:")
for g in sizes[sizes > 1].sort_values(ascending=False).index[:5]:
    print("  ", rwf.loc[rwf.group == g, "path"].tolist())

## Bước 4. Tách tập test chính thức, loại clip test trùng với train

- **Pool (train + val):** các video trong thư mục `train` của tác giả.
- **Test:** các video trong thư mục `val` của tác giả (đây là tập test chính thức của RWF-2000).
- Clip test giống với bất kỳ clip nào trong pool sẽ bị **loại khỏi test** để kết quả không bị thổi phồng. Danh sách clip bị loại được lưu lại để ghi vào báo cáo.

In [ ]:
is_test = rwf["orig_split"].str.lower() == OFFICIAL_TEST_FOLDER
pool_idx = np.where(~is_test)[0]
test_idx = np.where(is_test)[0]
print(f"Pool (train+val): {len(pool_idx)} | Test chính thức: {len(test_idx)}")
assert len(pool_idx) > 0 and len(test_idx) > 0, "Không tách được pool/test, kiểm tra OFFICIAL_TEST_FOLDER"

dist_tp = D[np.ix_(test_idx, pool_idx)]
dup_test = dist_tp.min(axis=1) <= THR
rwf_test_dropped = rwf.iloc[test_idx][dup_test].copy()
rwf_test_dropped["gan_nhat_trong_pool"] = [rwf.iloc[pool_idx[j]]["path"] for j in dist_tp[dup_test].argmin(axis=1)]
rwf_test_dropped["khoang_cach"] = dist_tp[dup_test].min(axis=1).astype(int)

rwf_test = rwf.iloc[test_idx][~dup_test].copy()
rwf_test["split"] = "test"
print(f"Clip test bị loại vì trùng với train/val: {int(dup_test.sum())}")
print(f"Test còn lại: {len(rwf_test)}")
display(rwf_test_dropped[["path", "gan_nhat_trong_pool", "khoang_cach"]].head(10)) if len(rwf_test_dropped) else None

## Bước 5. Chia train / val theo nhóm

Tách khoảng `VAL_RATIO` của pool làm validation. Chia **theo nhóm** (clip cùng nhóm cùng tập) và **phân tầng theo nhãn** (tỉ lệ đánh nhau / không đánh nhau giữ cân bằng).

In [ ]:
pool = rwf.iloc[pool_idx].copy()
n_splits = max(2, round(1 / VAL_RATIO))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

pool["split"] = "train"
val_pos = next(iter(sgkf.split(pool, pool["label"], pool["group"])))[1]   # fold đầu tiên làm val
pool.iloc[val_pos, pool.columns.get_loc("split")] = "val"

rwf_train = pool[pool.split == "train"].copy()
rwf_val = pool[pool.split == "val"].copy()

print(pd.crosstab(pd.concat([rwf_train, rwf_val, rwf_test])["split"],
                  pd.concat([rwf_train, rwf_val, rwf_test])["label"]).loc[["train", "val", "test"]].to_string())
print(f"\nTỉ lệ val trong pool: {len(rwf_val) / len(pool):.1%}")

## Bước 6. Loại clip RLVS / Hockey trùng với train/val của RWF-2000

Hai bộ này chỉ dùng để **test chéo**. Clip nào giống với train/val của RWF-2000 sẽ bị loại, vì mô hình đã "thấy" nó khi huấn luyện.

In [ ]:
pool_fp = fp_rwf[pool_idx]
external = {}
external_dropped = {}
for name in ("rlvs", "hockey"):
    if name not in data:
        continue
    d, f, _ = data[name]
    d = d.copy()
    dist = hamming(f, pool_fp)
    dup = dist.min(axis=1) <= THR
    dropped = d[dup].copy()
    dropped["gan_nhat_trong_rwf_pool"] = [rwf.iloc[pool_idx[j]]["path"] for j in dist[dup].argmin(axis=1)]
    dropped["khoang_cach"] = dist[dup].min(axis=1).astype(int)
    clean = d[~dup].copy()
    clean["split"] = "test"
    external[name], external_dropped[name] = clean, dropped
    print(f"[{name}] tổng {len(d)} | trùng với RWF train/val: {int(dup.sum())} | còn lại: {len(clean)}")
    print(pd.crosstab(clean["split"], clean["label"]).to_string(), "\n")

## Bước 7. Kiểm tra rò rỉ dữ liệu

Các dòng `assert` dưới đây phải chạy qua không báo lỗi. Nếu báo lỗi nghĩa là còn rò rỉ, quay lại chỉnh ngưỡng.

In [ ]:
# 1) Nhóm không bị chia vào cả train lẫn val
assert set(rwf_train["group"]).isdisjoint(set(rwf_val["group"])), "Có nhóm nằm ở cả train và val"

# 2) Không có đường dẫn nào xuất hiện ở hai tập
all_paths = pd.concat([rwf_train, rwf_val, rwf_test])["path"]
assert all_paths.is_unique, "Có video nằm ở hai tập"

# 3) Test không còn clip nào gần giống train/val
fp_trainval = fp_rwf[pool_idx]
fp_test_final = fp_rwf[rwf_test.index.to_numpy()]
min_d = hamming(fp_test_final, fp_trainval).min()
assert min_d > THR, f"Test vẫn còn clip giống train/val (khoảng cách nhỏ nhất {min_d})"
print(f"RWF: khoảng cách nhỏ nhất giữa test và train/val = {int(min_d)} (ngưỡng {THR})")

# 4) Bộ test chéo không còn clip giống train/val của RWF
for name, clean in external.items():
    f = data[name][1][clean.index.to_numpy()]
    m = hamming(f, fp_trainval).min()
    assert m > THR, f"{name} vẫn còn clip trùng (khoảng cách nhỏ nhất {m})"
    print(f"{name}: khoảng cách nhỏ nhất đến train/val của RWF = {int(m)} (ngưỡng {THR})")
print("\nKiểm tra rò rỉ: ĐẠT")

## Bước 8. Lưu file CSV

In [ ]:
COLS = ["dataset", "path", "abs_path", "label", "split", "orig_split", "frames", "fps", "width", "height", "duration_s"]
outputs = {"rwf_train": rwf_train, "rwf_val": rwf_val, "rwf_test": rwf_test}
for name, clean in external.items():
    outputs[f"{name}_test"] = clean

for fname, df_out in outputs.items():
    df_out[COLS].to_csv(OUT_DIR / f"{fname}.csv", index=False)
rwf_test_dropped.to_csv(OUT_DIR / "rwf_test_dropped_duplicates.csv", index=False)
for name, dropped in external_dropped.items():
    dropped.to_csv(OUT_DIR / f"{name}_dropped_duplicates.csv", index=False)
bad_all.to_csv(OUT_DIR / "bad_videos.csv", index=False)

summary = pd.DataFrame([
    dict(tap=k, so_clip=len(v), danh_nhau=int((v.label == 1).sum()), khong_danh_nhau=int((v.label == 0).sum()))
    for k, v in outputs.items()
])
summary.to_csv(OUT_DIR / "summary.csv", index=False)
print(summary.to_string(index=False))
print("\nĐã lưu vào:", OUT_DIR.resolve())

In [ ]:
# Biểu đồ cân bằng nhãn giữa các tập
ax = summary.set_index("tap")[["danh_nhau", "khong_danh_nhau"]].plot(kind="bar", figsize=(8, 3.5))
ax.set_ylabel("Số clip"); ax.set_title("Số clip theo tập và nhãn"); plt.xticks(rotation=0); plt.show()

## Bước 9. Kiểm tra thử đọc frame

Lấy ngẫu nhiên vài video từ mỗi tập, đọc 8 frame đều nhau để chắc chắn dữ liệu đọc được và nhãn đúng.

In [ ]:
def sample_frames(path, n=8, size=(112, 112)):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    for idx in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, fr = cap.read()
        frames.append(cv2.cvtColor(cv2.resize(fr, size), cv2.COLOR_BGR2RGB) if ok else np.zeros((*size, 3), np.uint8))
    cap.release()
    return frames

rng = np.random.default_rng(0)
picks = []
for tap in ("rwf_train", "rwf_val", "rwf_test"):
    df_t = outputs[tap]
    for lab in (1, 0):
        sub = df_t[df_t.label == lab]
        if len(sub):
            picks.append((tap, lab, sub.iloc[rng.integers(len(sub))]["abs_path"]))

fig, axes = plt.subplots(len(picks), 8, figsize=(14, 1.9 * len(picks)))
axes = np.atleast_2d(axes)
for r, (tap, lab, p) in enumerate(picks):
    for c, fr in enumerate(sample_frames(p)):
        axes[r, c].imshow(fr); axes[r, c].axis("off")
    axes[r, 0].set_title(f"{tap} | {'ĐÁNH NHAU' if lab == 1 else 'bình thường'}", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

## Kết thúc

Thư mục `manifests/` đã có:

| File | Dùng để |
|---|---|
| `rwf_train.csv` | Huấn luyện |
| `rwf_val.csv` | Chọn mô hình, chọn ngưỡng cảnh báo |
| `rwf_test.csv` | Test chính |
| `rlvs_test.csv`, `hockey_test.csv` | Test chéo |
| `*_dropped_duplicates.csv`, `bad_videos.csv` | Ghi vào báo cáo (số clip bị loại và lý do) |

Bước tiếp theo: viết `Dataset` đọc các file CSV này và huấn luyện baseline CNN-BiLSTM.